# boolean-mask-combine composite — cx29: combine non-singular det mask with in-range solve mask

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `singular-matrix-mask-trick`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "singular-matrix-mask-trick"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "Numpy: Singular matrix mask trick"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `raytrace_triangle` has to solve `(B, 3, 3)` systems where SOME slices are exactly singular (parallel ray, degenerate triangle). `t.linalg.solve` raises if ANY slice is singular — so we use the **singular-matrix-mask trick**:
1. Compute `dets = t.linalg.det(A)`.
2. Build `is_singular = dets.abs() < eps` — boolean of shape `(B,)`.
3. **Mask in** the identity matrix at singular slices: `A_safe = A.clone(); A_safe[is_singular] = t.eye(N)`. Now `solve` succeeds everywhere.
4. Run the solve on `A_safe`. The values at singular slices are garbage.
5. **Mask out** those garbage values with `valid = (~is_singular) & in_range`.

The composition: this drill exercises both the **mask-in** half (overwrite singular A's with the identity) and the **mask-out** half (boolean-AND of `~is_singular` with the downstream `in_range` predicate).

**Anatomy.**
- `nonsingular = ~is_singular` — the per-slice gating mask.
- `in_range = (x >= 0).all(dim=-1) & (x <= 1).all(dim=-1)` — barycentric range check.
- `valid = nonsingular & in_range` — boolean-AND the two masks.

### Composite Exercise — combine non-singular det mask with in-range solve mask

**Atoms exercised together**: `boolean-mask-combine`, `singular-matrix-mask-trick`

Implement `cx29_safe_solve(A, b, eps=1e-8)`.

- `A`: float tensor of shape `(B, N, N)`. May include slices with `det(A) == 0` (parallel rays / degenerate triangles).
- `b`: float tensor of shape `(B, N)`.
- `eps`: tolerance for the singular test.

Return `(x, valid)`:
- `x`: solve result for the masked-in `A`, shape `(B, N)`. At singular slices the values are arbitrary (we mask them out, not zero them out — keeps the downstream code branch-free).
- `valid`: boolean tensor of shape `(B,)`, True iff the slice is non-singular AND `x[i]` is entirely in `[0, 1]`.

1. **Mask-in trick** — compute `is_singular = t.linalg.det(A).abs() < eps`, then overwrite the singular slices of a COPY of `A` with `t.eye(N)`. Do NOT mutate the caller's `A`.
2. **Solve** the masked-in system.
3. **Mask combine** — AND `~is_singular` with the in-range predicate.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx29_safe_solve(A, b, eps=1e-8):
    raise NotImplementedError

def _test_cx29():
    # Case A: clean 2x2 batch, all non-singular, all in-range.
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[2.0, 0.0], [0.0, 2.0]],
    ])
    b = t.tensor([[0.3, 0.4], [0.5, 0.5]])
    x, valid = cx29_safe_solve(A, b)
    assert tuple(x.shape) == (2, 2)
    assert tuple(valid.shape) == (2,)
    assert valid.dtype == t.bool
    assert valid.tolist() == [True, True]
    assert t.allclose(x, t.tensor([[0.3, 0.4], [0.25, 0.25]]), atol=1e-5)

    # Case B: one slice is EXACTLY singular (det == 0). The mask-in trick must prevent a crash.
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[1.0, 1.0], [1.0, 1.0]],  # rank-1, det = 0.
        [[1.0, 0.0], [0.0, 1.0]],
    ])
    b = t.tensor([[0.2, 0.3], [99.0, 99.0], [0.5, 0.5]])
    # This MUST NOT raise. If it does, the singular-matrix-mask trick wasn't applied.
    x, valid = cx29_safe_solve(A, b)
    assert tuple(x.shape) == (3, 2)
    assert valid.tolist() == [True, False, True], f'got {valid.tolist()}'
    # At non-singular slices, x must equal b (identity A's).
    assert t.allclose(x[0], b[0])
    assert t.allclose(x[2], b[2])

    # Case C: caller's A must not be mutated by the mask-in step.
    A_orig = t.tensor([
        [[1.0, 1.0], [1.0, 1.0]],  # singular.
        [[1.0, 0.0], [0.0, 1.0]],
    ])
    A = A_orig.clone()
    b = t.tensor([[1.0, 1.0], [0.5, 0.5]])
    x, valid = cx29_safe_solve(A, b)
    assert t.equal(A, A_orig), 'cx29 must not mutate the caller A — clone before masking-in'

    # Case D: out-of-range solve at a non-singular slice still gets masked out.
    A = t.tensor([
        [[1.0, 0.0], [0.0, 1.0]],
        [[1.0, 0.0], [0.0, 1.0]],
    ])
    b = t.tensor([[0.3, 0.4], [2.5, -0.5]])
    x, valid = cx29_safe_solve(A, b)
    assert valid.tolist() == [True, False]
    _dd_passed.add('cx29')

_test_cx29()

<details><summary>Show solution — cx29</summary>

```python
def cx29_safe_solve(A, b, eps=1e-8):
    B, N, _ = A.shape
    # Atom A (singular-matrix-mask-trick): detect singular slices and mask in the identity
    # on a COPY so the caller's A is preserved.
    dets = t.linalg.det(A)
    is_singular = dets.abs() < eps
    A_safe = A.clone()
    A_safe[is_singular] = t.eye(N, dtype=A.dtype, device=A.device)
    x = t.linalg.solve(A_safe, b)
    # In-range predicate over the last axis.
    in_range = (x >= 0).all(dim=-1) & (x <= 1).all(dim=-1)
    # Atom B (boolean-mask-combine): AND the (~singular) mask with the in-range predicate.
    valid = (~is_singular) & in_range
    return x, valid
```

The `A.clone()` is critical: without it, the caller's tensor gets a bunch of identity rows stamped over its singular slices — a silent state corruption bug. The `~is_singular & in_range` AND is the canonical ARENA shape: one mask says "the math was valid", the other says "the result is physically meaningful".
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["Numpy: Boolean mask combine", "Numpy: Singular matrix mask trick"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()